# Quantum Probability Simulator

**Day 20 | 30-Day Quantum Computing Challenge**

## Aim:
To simulate repeated quantum measurements and compare the probabilities of 0 and 1 for different shot counts.

## Tools:
Python, Qiskit, Qiskit Aer, Matplotlib

## Expected Result:
The measured proportions generally become more stable around 50% as the number of shots increases.

**Execution:** Run the cells from top to bottom. The notebook uses an ideal simulator unless explicitly stated otherwise.

## 1. Concept and circuit

Prepare $|0\rangle$, apply the Hadamard gate and measure.

```text
|0> --- H --- Measure
```

The state becomes $|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}$, so the **theoretical** probabilities are $P(0)=P(1)=0.5$. A finite sample does not have to be exactly 50/50.

**Important:** The default experiment uses NumPy to sample the *ideal measurement distribution*. It is a reproducible classical simulation of the quantum measurement probabilities, not execution on quantum hardware.

In [ ]:
# Experiment step: execute this cell and inspect its output.
import numpy as np
import matplotlib.pyplot as plt

SHOTS = [10, 100, 1_000, 10_000]
SEED = 2026
rng = np.random.default_rng(SEED)
print("Theoretical probabilities: P(0)=0.5, P(1)=0.5")

## 2. Independent experiments for each shot count
Each row is a separate set of repeated measurements. The observed percentage may lie above or below 50%.

In [ ]:
# Experiment step: execute this cell and inspect its output.
results = {}
for n in SHOTS:
    samples = rng.binomial(n=1, p=0.5, size=n)
    zeros = int(np.count_nonzero(samples == 0))
    ones = n - zeros
    results[n] = {"samples": samples, "zeros": zeros, "ones": ones, "p0": zeros/n, "p1": ones/n}
    print(f"{n:>6,} shots | 0: {zeros:>5,} ({zeros/n:6.2%}) | 1: {ones:>5,} ({ones/n:6.2%})")

## 3. Visual comparison: 10 vs 100 vs 1,000 vs 10,000

In [ ]:
# Experiment step: execute this cell and inspect its output.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout="constrained")
x = np.arange(len(SHOTS))
p0 = [results[n]["p0"]*100 for n in SHOTS]
p1 = [results[n]["p1"]*100 for n in SHOTS]
axes[0].bar(x-0.18, p0, width=0.36, color="#3268aa", label="0")
axes[0].bar(x+0.18, p1, width=0.36, color="#9a7bd3", label="1")
axes[0].axhline(50, color="#d08036", linestyle="--", label="Theory: 50%")
axes[0].set_xticks(x, [str(n) for n in SHOTS])
axes[0].set(xlabel="Shots", ylabel="Observed percentage (%)", ylim=(0, 100), title="Independent experiments")
axes[0].legend()

## 4. Cumulative convergence
For a more meaningful convergence plot, use **one 10,000-shot sequence** and plot its running frequency. The law of large numbers says the observed frequency tends toward the underlying probability as the number of independent shots grows; it does not require a smooth or monotonic approach.

In [ ]:
# Experiment step: execute this cell and inspect its output.
rng_continuous = np.random.default_rng(SEED)
sequence = rng_continuous.binomial(n=1, p=0.5, size=10_000)
shot_index = np.arange(1, len(sequence)+1)
running_p0 = np.cumsum(sequence == 0) / shot_index
axes[1].plot(shot_index, running_p0*100, color="#3268aa", lw=1.2, label="Running P(0)")
axes[1].axhline(50, color="#d08036", linestyle="--", label="Theory: 50%")
axes[1].set_xscale("log")
axes[1].set(xlim=(1, 10_000), ylim=(0, 100), xlabel="Cumulative shots (log scale)", ylabel="Observed percentage (%)", title="One continuous experiment")
axes[1].legend()
fig.suptitle("Quantum Probability Simulator | Day 20", fontsize=15, fontweight="bold")
fig.savefig("results.png", dpi=190, bbox_inches="tight")
plt.show()

## 5. Statistical interpretation
For $n$ independent fair measurements, the number of zeros follows $K_0\sim\mathrm{Binomial}(n,0.5)$. The standard deviation of the **observed fraction** is $\sqrt{0.5(1-0.5)/n}=1/(2\sqrt{n})$. Therefore, typical sampling fluctuations shrink with larger $n$ (approximately proportional to $1/\sqrt{n}$).

In [ ]:
# Experiment step: execute this cell and inspect its output.
print("Typical one-standard-deviation fluctuations in observed P(0):")
for n in SHOTS:
    print(f"{n:>6,} shots: ±{100*0.5/np.sqrt(n):.2f} percentage points")

## 6. Optional: execute the actual circuit using Qiskit Aer
Run the following cell only after installing `qiskit` and `qiskit-aer` (`%pip install qiskit qiskit-aer`). The cell gracefully skips the optional step if the packages are unavailable. Circuit simulation is still not physical quantum hardware.

In [ ]:
# Experiment step: execute this cell and inspect its output.
try:
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    print(qc.draw(output="text"))
    simulator = AerSimulator(seed_simulator=SEED)
    for n in SHOTS:
        counts = simulator.run(qc, shots=n).result().get_counts()
        print(f"{n:>6,} shots: {counts}")
except ImportError:
    print("Optional Qiskit experiment skipped. Install qiskit and qiskit-aer to run it.")

## 7. Experiment conclusions
- More shots generally reduce the *typical size* of sampling error; they do not guarantee that every larger sample is closer to 50%.
- The **bar chart** compares separate experiments; the **running curve** shows convergence within a single experiment.
- The theoretical distribution is 50/50 because of the Hadamard-prepared state.
- This project simulates ideal measurement statistics, not quantum advantage or certified physical randomness.

**Next:** Day 21 — Quantum Random Number Generator.

## Observation:
Record the measured or simulated outputs and compare them with the theoretical expectation. Note that finite-shot results can fluctuate.

## Conclusion:
The experiment demonstrates the quantum circuit or state transformation described above using simulation; it does not establish performance on physical quantum hardware.